In [3]:
%pip install -q "transformers==4.57.1"
%pip install -q --no-deps "bitsandbytes==0.48.2"

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
import gc
import hashlib
import json
import re
import unicodedata
from decimal import Decimal
from pathlib import Path
from time import perf_counter

import bitsandbytes as bnb
import pandas as pd
import torch
from PIL import Image
from peft import PeftModel, prepare_model_for_kbit_training
from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    GenerationConfig,
    Qwen3VLForConditionalGeneration
)

In [6]:
def read_json(path):
    return path.read_text(encoding="utf-8") and json.loads(
        path.read_text(encoding="utf-8")
    )

def save_json(path, data):
    tmp = path.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    tmp.replace(path)

def sha256(path):
    with path.open("rb") as file:
        return hashlib.file_digest(file, "sha256").hexdigest()

In [7]:
ROOT = Path("/content/drive/MyDrive/MLLM/mordern_llm")
DATA = ROOT / "data/sroie"
BEST = ROOT / "outputs/qlora_run1/best_adapter"
OUT = ROOT / "outputs/evaluation/day5"
OUT.mkdir(parents=True, exist_ok=True)

recipe = read_json(BEST / "training_recipe.json")
base_cfg = recipe["base"]
task_config = read_json(BEST / "task_config.json")
manifest_path = DATA / "manifest.json"
manifest = read_json(manifest_path)

DOC_PROMPT = task_config["prompt"]
IMAGE_KWARGS = task_config["image_kwargs"]
FIELDS = tuple(task_config["fields"])

test_records = manifest["test"]
test_ids = [r["id"] for r in test_records]

saved_example = read_json(BEST / "validation_rows.json")[0]
warmup_record = next(r for r in manifest["val"] if r["id"] == saved_example["id"])

DEVICE = torch.device("cuda:0")
processor = AutoProcessor.from_pretrained(BEST, use_fast=recipe["processor_use_fast"])

run_config = {
    "base": base_cfg,
    "task": task_config,
    "manifest_sha256": sha256(manifest_path),
    "adapter_sha256": sha256(BEST / "adapter_model.safetensors"),
    "gpu": torch.cuda.get_device_name(DEVICE),
    "torch_version": str(torch.__version__)
}

print("Fixed test receipts:", len(test_ids))
print("Selected epoch:", read_json(BEST / "best_metrics.json")["epoch"])

Fixed test receipts: 100
Selected epoch: 1.0


In [8]:
def load_base(for_training):
    quant_config = BitsAndBytesConfig.from_dict(
        base_cfg["quantization_config"]
    )
    net = Qwen3VLForConditionalGeneration.from_pretrained(
        base_cfg["model_id"],
        revision=base_cfg["revision"],
        quantization_config=quant_config,
        dtype=torch.float16,
        device_map={"": 0},
        attn_implementation="sdpa",
        low_cpu_mem_usage=True
    )
    
    net = prepare_model_for_kbit_training(
        net,
        use_gradient_checkpointing=for_training,
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
    
    net.model.visual.to(dtype=torch.float16)
    net.config.use_cache = not for_training
    net.config.text_config.use_cache = not for_training
    return net


def normal(field, value):
    if not isinstance(value, str) or not value.strip():
        return None

    text = " ".join(
        unicodedata.normalize("NFKC", value).casefold().split()
    )
    if field != "total":
        return text
    
    text = re.sub(r"^(?:rm|\$)\s*", "", text)
    if not re.fullmatch(
        r"[+-]?(?:\d+|\d{1,3}(?:,\d{3})+)(?:\.\d+)?",
        text
    ):
        return None

    amount = Decimal(text.replace(",", ""))
    return format(amount.normalize(), "f")

In [9]:
def score_doc(raw, reference):
    try:
        json.loads(raw)
        raw_valid = True
    except ValueError:
        raw_valid = False
    
    text = raw.strip()
    lines = text.splitlines()
    wrapped = (
        len(lines) >= 3
        and lines[0].strip().lower() in ("```json", "```")
        and lines[-1].strip() == "```"
    )
    
    if wrapped:
        text = "\n".join(lines[1:-1])
    
    try:
        prediction = json.loads(text)
    except ValueError:
        prediction = None

    schema = (
        isinstance(prediction, dict)
        and set(prediction) == set(FIELDS)
        and all(
            isinstance(value, str) or value is None
            for value in prediction.values()
        )
    )
    correct = {}
    for field in FIELDS:
        if not schema:
            correct[field] = False
        elif prediction[field] is None or reference[field] is None:
            correct[field] = prediction[field] is reference[field]
        else:
            pred = normal(field, prediction[field])
            gold = normal(field, reference[field])
            correct[field] = pred is not None and pred == gold

    return {
        "prediction": prediction,
        "raw_json_valid": raw_valid,
        "schema_valid": schema,
        "used_fence_cleanup": wrapped,
        **{field + "_correct": value for field, value in correct.items()},
        "all_correct": all(correct.values())
    }

In [10]:
def read_rgb(path):
    with Image.open(path) as source:
        return source.convert("RGB")

def receipt_image(record):
    path = Path(record["image_path"])
    if not path.is_file():
        path = DATA / "images" / path.name
    return read_rgb(path)

generation = GenerationConfig(
    max_new_tokens=task_config["max_new_tokens"],
    do_sample=False,
    use_cache=True,
    eos_token_id=processor.tokenizer.eos_token_id,
    pad_token_id=processor.tokenizer.pad_token_id,
    bos_token_id=processor.tokenizer.bos_token_id
)

@torch.no_grad()
def answer_image(net, image, prompt=DOC_PROMPT):
    messages = [{
        "role" : "user",
        "content" : [
            {"type": "image"},
            {"type": "text", "text": prompt}
        ]
    }]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    batch = processor(
        text=[text],
        images=[image],
        images_kwargs=IMAGE_KWARGS,
        add_special_tokens=False,
        return_tensors="pt"
    ).to(DEVICE)
    
    with torch.autocast("cuda", dtype=torch.float16):
        output = net.generate(
            **batch,
            generation_config=generation,
            use_model_defaults=False
        )
    
    new_ids = output[:, batch["input_ids"].shape[1]:]
    return processor.batch_decode(
        new_ids, skip_special_tokens=True
    )[0].strip()

In [ ]:
IMAGE_DIR = Path("/content/drive/MyDrive/MLLM/data/images")
def find_image(prefix):
    matches = [
        p for p in IMAGE_DIR.glob(f"{prefix}*")
        if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
    ]

    return str(matches[0])

flower = find_image("0c1be4")
city = find_image("c7fa0a")
food = find_image("335eb4")

VQA_CASES = [
    {
        "id": "flower_color",
        "image_path": flower,
        "question": "What color is the flower? Answer in one word.",
        "reference": "yellow"
    },
    {
        "id": "flower_subject",
        "image_path": flower,
        "question": "Which is the main subject, a flower or a car? Answer flower or car.",
        "reference": "flower"
    },
    {
        "id": "city_viewpoint",
        "image_path": city,
        "question": "Is the city viewed from above or from street level? Answer above or street level.",
        "reference": "above"
    },
    {
        "id": "city_street_level",
        "image_path": city,
        "question": "Was this photo taken at street level? Answer only yes or no.",
        "reference": "no"
    },
    {
        "id": "food_subject",
        "image_path": food,
        "question": "Which is the main subject, food or a vehicle? Answer food or vehicle.",
        "reference": "food"
    },
]

print("VQA questions:", len(VQA_CASES))
print("Distinct Drive images:", len({r["image_path"] for r in VQA_CASES}))

VQA questions: 5
Distinct Drive images: 3


**Test evaluation function**

In [13]:
def evaluate_receipts(net, setting):
    path = OUT / f"{setting}_receipts.json"
    state = (
        read_json(path) if path.exists()
        else {"config": run_config, "rows": []}
    )
    
    rows = state["rows"]
    done = len(rows)
    
    for record in test_records[done:]:
        image = receipt_image(record)
        
        torch.cuda.synchronize(DEVICE)
        torch.cuda.reset_peak_memory_stats(DEVICE)
        start = perf_counter()
        
        answer = answer_image(net, image)
        torch.cuda.synchronize(DEVICE)
        latency = perf_counter() - start
        peak = torch.cuda.max_memory_allocated(DEVICE) / 2**30
        
        rows.append({
            "id": record["id"],
            "reference": record["reference"],
            "answer": answer,
            **score_doc(answer, record["reference"]),
            "latency_s": latency,
            "peak_gpu_GiB": peak
        })
        
        if len(rows) % 10 == 0 or len(rows) == len(test_records):
            save_json(path, state)
            print(f"{setting}: {len(rows)}/{len(test_records)} saved")

    return rows


def evaluate_vqa(net):
    return [
        {
            **case,
            "answer": answer_image(
                net, read_rgb(case["image_path"]), case["question"]
            )
        }
        for case in VQA_CASES
    ]

**Base → adapter → adapted**